# Pipeline for model evaluation for the graph data extraction task

Import bibs

In [2]:
import json
import pandas as pd
import torch
from neo4j import GraphDatabase
from bert_score import score as bert_scorer
from langchain_openai import ChatOpenAI
from langchain_community.llms import LlamaCpp
from llama_cpp import Llama
from langchain_huggingface import HuggingFacePipeline
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os

Initial Configuration

In [3]:
# --- Load environment variable

load_dotenv()

uri = os.getenv("NEO4J_URI")
username = os.getenv("NEO4J_USERNAME")
password = os.getenv("NEO4J_PASSWORD")
openai_api_key = os.getenv("OPENAI_API_KEY")

# --- QA File path

QA_FILE_PATH = "../data/qa_neo4j.json"
RESULTS_CSV_PATH = "../data/evaluation_results_revised-HF27B_v2.csv"

# --- Choice of model to avaliate

#OpenAI models

model = "gpt-4.1-mini"  

# --- Model who will be the judge for llm-as-a-judge
JUDGE_LLM = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

Defining some functions 

In [10]:
# -- Connect to the graph

def ConnectToGraph(uri, username, password):
    try:
        graph = Neo4jGraph(
            url=uri,
            username=username,
            password=password
        )

    except Exception as e:
        print(f"Erro ao conectar-se ao Neo4j: {e}")
        graph = None
        exit()

    return graph

# --- Creation of the chain

def CreateChain(graph, llm, CYPHER_GENERATION_PROMPT, QA_PROMPT):
    """
    Cria uma cadeia de perguntas e respostas usando Cypher para o grafo Neo4j.
    """
    cypher_chain = GraphCypherQAChain.from_llm(
        llm=llm,
        graph=graph,
        cypher_prompt=CYPHER_GENERATION_PROMPT,
        qa_prompt=QA_PROMPT,
        allow_dangerous_requests=True,
        validate_cypher=True,
        return_intermediate_steps = True
    )
    return cypher_chain

# --- Creation of LLM model

def CreateLLMOpenai(openai_api_key):
    """
    Cria um modelo de linguagem usando o modelo de preferência.
    """
    llm = ChatOpenAI(model=model, temperature=0, api_key=openai_api_key) 

    CYPHER_GENERATION_TEMPLATE = """
    Task: You are an expert Neo4j Cypher translator. Given a graph schema and a user question in Portuguese, generate a Cypher query to answer the question.
    
    Instructions:
    - Use only the provided relationship types and properties in the schema.
    - Do not use any other relationship types or properties that are not provided.
    - DO NOT answer the question, ONLY generate the Cypher query.
    - Do not use examples in the query.
    - Do not put the query in Markdown code blocks.
    - Use the `toLower()` function for case-insensitive string comparisons.
    - Use `CONTAINS` for partial string matching.
    - Pay attention to acronyms (e.g., ITA is 'Instituto Tecnológico de Aeronáutica').
    - Always use a `RETURN` statement to provide the requested data.


    Schema:
    {schema}

    Question: {query}
    Query Cypher:
    """
    CYPHER_GENERATION_PROMPT = PromptTemplate(
        input_variables=["schema", "question"], template=CYPHER_GENERATION_TEMPLATE
    )

    # Este é o prompt para gerar a resposta final com base nos resultados da query
    QA_TEMPLATE = """
    Task: You are an AI assistant that summarizes the results of a graph database query to answer a question.
    The user's question and the query results are provided below.
    Formulate a clear and natural answer in PORTUGUESE.

    **Instructions:**
    1.  If the context list is empty (`[]`), you MUST reply EXACTLY with: 'Não encontrei informações sobre isso no banco de dados.'
    2.  Otherwise, read the keys and values from the JSON context and use them to construct a fluent, helpful answer in PORTUGUESE.
    3.  If there are multiple results, combine them naturally. For example, if the context is `[{{"Projeto": "Sistema X"}}, {{"Projeto": "Sistema Y"}}]`, a good answer is "Os projetos encontrados foram Sistema X e Sistema Y."

    Context:
    {context}

    Question: {query}
    Helpful Answer in Portuguese: """


    QA_PROMPT = PromptTemplate(input_variables=["context", "question"], template=QA_TEMPLATE)
    
    return llm, CYPHER_GENERATION_PROMPT, QA_PROMPT

def CreateLLMHuggingFace():
    """
    Cria um modelo de linguagem usando o modelo de preferência.
    """
    llm = ChatOpenAI(
    
    base_url="http://127.0.0.1:8080/v1",
    
    api_key="", 
    
    temperature=0
)

    CYPHER_GENERATION_TEMPLATE = """
    Task: You are an expert Neo4j Cypher translator. Given a graph schema and a user question in Portuguese, generate a Cypher query to answer the question.
    
    Instructions:
    - Use only the provided relationship types and properties in the schema.
    - Do not use any other relationship types or properties that are not provided.
    - DO NOT answer the question, ONLY generate the Cypher query.
    - Do not use examples in the query.
    - Do not put the query in Markdown code blocks.
    - Use the `toLower()` function for case-insensitive string comparisons.
    - Use `CONTAINS` for partial string matching.
    - Pay attention to acronyms (e.g., ITA is 'Instituto Tecnológico de Aeronáutica').
    - Always use a `RETURN` statement to provide the requested data.


    Schema:
    {schema}

    Question: {query}
    Query Cypher:
    """
    CYPHER_GENERATION_PROMPT = PromptTemplate(
        input_variables=["schema", "query"], template=CYPHER_GENERATION_TEMPLATE
    )

    # Este é o prompt para gerar a resposta final com base nos resultados da query
    QA_TEMPLATE = """
    Task: You are an AI assistant that summarizes the results of a graph database query to answer a question.
    The user's question and the query results are provided below.
    Formulate a clear and natural answer in PORTUGUESE.

    **Instructions:**
    1. If the context list is empty (`[]`), you MUST reply EXACTLY with: 'Não encontrei informações sobre isso no banco de dados.'
    2. Otherwise, each element of the context will be a dictionary. Frequently, the structure will look like `[{"p": {...}}, {"p": {...}}]`.
       - In this case, you must look inside the key `"p"` to access the actual properties of the node, such as:
         - `nomePessoa`
         - `cargoPessoa`
         - `patentePessoa`
         - `dataAdmissaoPessoa`
         - `emailPessoa`
         - `statusPessoa`
    3. Use these properties to construct a fluent, helpful answer in PORTUGUESE.
    4. If there are multiple results, combine them naturally. For example, if the context is
       `[{"p": {"nomePessoa": "Militar 7"}}, {"p": {"nomePessoa": "Militar 14"}}]`,
       a good answer is: "As pessoas encontradas foram Militar 7 e Militar 14."
    Context:
    {context}

    Question: {query}
    Helpful Answer in Portuguese: """


    QA_PROMPT = PromptTemplate(input_variables=["context", "query"], template=QA_TEMPLATE)
    
    return llm, CYPHER_GENERATION_PROMPT, QA_PROMPT

Functions of assessment

In [5]:
def can_cypher_execute(query: str, driver) -> bool:
    """Verifica se a query Cypher é executável (sintaxe correta)."""
    if not query or query.strip() == "":
        return False
    try:
        with driver.session() as session:
            session.run(f"EXPLAIN {query}")
        return True
    except Exception:
        return False
    
def evaluate_factual_consistency(question: str, gold_answer: str, generated_answer: str, judge_llm) -> str:
    """
    Usa um LLM como juiz para verificar a consistência factual.
    """
    prompt_template = """Você é um avaliador preciso e rigoroso. Sua tarefa é comparar uma 'Resposta Gerada' com uma 'Resposta de Referência (Fonte da Verdade)' no contexto de uma 'Pergunta Original'.

    Determine se a 'Resposta Gerada' é factualmente consistente com a 'Resposta de Referência'. A redação não precisa ser idêntica, mas os fatos, nomes, números e conclusões principais devem ser os mesmos.

    Responda com APENAS UMA das seguintes categorias:
    - CONSISTENTE: A resposta gerada afirma os mesmos fatos que a resposta de referência.
    - INCONSISTENTE: A resposta gerada contradiz ou afirma fatos diferentes da resposta de referência.
    - PARCIALMENTE CONSISTENTE: A resposta gerada acerta alguns fatos, mas omite outros ou contém informações adicionais não verificáveis.

    Pergunta Original:
    {question}

    Resposta de Referência (Fonte da Verdade):
    {gold_answer}

    Resposta Gerada:
    {generated_answer}

    Categoria de Consistência Factual:"""

    eval_prompt = PromptTemplate(
        input_variables=["question", "gold_answer", "generated_answer"],
        template=prompt_template
    )

    eval_chain = eval_prompt | judge_llm | StrOutputParser()
    
    try:
        response = eval_chain.invoke({
            "question": question,
            "gold_answer": gold_answer,
            "generated_answer": generated_answer
        })
        return response.strip().upper()
    except Exception as e:
        print(f"Erro na avaliação pelo LLM Juiz: {e}")
        return "ERRO_AVALIACAO"

In [6]:
# --- PIPELINE PRINCIPAL ---

graph = ConnectToGraph(uri,username,password)
llm, cypher_prompt, qa_prompt = CreateLLMHuggingFace()
gpt,_,_ = CreateLLMOpenai(openai_api_key)
chain = CreateChain(graph,llm,cypher_prompt,qa_prompt)
# chainGPT = CreateChain(graph,gpt,cypher_prompt,qa_prompt)

def run_evaluation():
    with open(QA_FILE_PATH, 'r', encoding='utf-8') as f:
        qa_dataset = json.load(f)

    neo4j_driver = GraphDatabase.driver(uri, auth=(username, password))
    evaluation_results = []
    total_items = len(qa_dataset)

    print("Iniciando o pipeline de avaliação revisado...")

    for i, item in enumerate(qa_dataset):
        print(f"Processando questão {i+1}/{total_items}: \"{item['question']}\"")

        question = item['question']
        gold_cypher = item['gold_cypher']
        gold_answer = item['gold_answer']

        try:
            raw_cypher = chain.cypher_generation_chain.invoke({"schema": chain.graph.get_schema, "query": question})
            generated_cypher = raw_cypher.strip().replace('\\n', ' ')
            context = neo4j_driver.session().run(generated_cypher).data()
            generated_answer = chain.qa_chain.invoke({"context": context, "query": question})

            # 1. Avaliar a query Cypher (apenas se executa)
            cypher_executable = can_cypher_execute(generated_cypher, neo4j_driver)

            # 2. Avaliar a resposta textual
            P, R, F1 = bert_scorer([generated_answer], [gold_answer], lang="pt", model_type='bert-base-multilingual-cased')
            bert_f1 = F1.item()

            # factual_consistency = evaluate_factual_consistency(question, gold_answer, generated_answer, JUDGE_LLM)

            evaluation_results.append({
                "id": item['id'],
                "question": question,
                "gold_cypher": gold_cypher,
                "generated_cypher": generated_cypher,
                "cypher_executable": cypher_executable,
                "gold_answer": gold_answer,
                "generated_answer": generated_answer,
                "bert_score_f1": bert_f1,
                # "factual_consistency": factual_consistency
            })
        
        except Exception as e:
            print(f"!!! ERRO GERAL ao processar a questão ID {item['id']}: {e}")
            evaluation_results.append({
                "id": item['id'], "question": question, "generated_cypher": "ERRO",
                "cypher_executable": False, "generated_answer": f"ERRO NA EXECUÇÃO: {e}",
                "bert_score_f1": 0.0, "factual_consistency": "ERRO_EXECUCAO"
            })

    neo4j_driver.close()
    df = pd.DataFrame(evaluation_results)
    df.to_csv(RESULTS_CSV_PATH, index=False, encoding='utf-8-sig')
    print(f"\nResultados detalhados salvos em: {RESULTS_CSV_PATH}")

    # --- ANÁLISE DOS RESULTADOS ---
    cypher_exec_rate = df['cypher_executable'].mean() * 100
    average_bert_score = df['bert_score_f1'].mean()
    consistency_distribution = df['factual_consistency'].value_counts(normalize=True) * 100
    
    print("\n--- RESUMO DA AVALIAÇÃO REVISADA ---")
    print(f"Taxa de Execução das Queries Cypher: {cypher_exec_rate:.2f}%")
    print(f"BERTScore F1 Médio (Similaridade Semântica): {average_bert_score:.4f}")
    print("\nDistribuição da Consistência Factual:")
    print(consistency_distribution.to_string())
    print("-------------------------------------\n")
    
    print("Amostra de Respostas marcadas como INCONSISTENTES:")
    inconsistent_df = df[df['factual_consistency'] == 'INCONSISTENTE']
    print(inconsistent_df[['id', 'question', 'gold_answer', 'generated_answer']].head())

ChatGPT 4o mini - Cost: $0.5 - Time: 7m 25.7s

In [ ]:
run_evaluation()

Iniciando o pipeline de avaliação revisado...
Processando questão 1/95: "Quais Pessoas com patente de Major possuem a competência 'Gestão de Projetos'?"


ChatGPT 4 - Cost: $7.98 - Time: 24m 48.5s

In [ ]:
run_evaluation()

Iniciando o pipeline de avaliação revisado...
Processando questão 1/95: "Quais militares com patente de Major possuem a competência 'Gestão de Projetos'?"
Processando questão 2/95: "Quais projetos utilizam a tecnologia PostgreSQL e estão em andamento?"
Processando questão 3/95: "Quais militares com patente de Coronel são responsáveis por documentação?"
Processando questão 4/95: "Quais projetos concluídos possuem documentação crítica de nível ALTA?"
Processando questão 5/95: "Quais militares possuem certificação PMP?"
Processando questão 6/95: "Quais treinamentos foram concluídos por militares de patente Capitão?"
Processando questão 7/95: "Quais competências são requeridas pelo projeto 'Simulador de Voo IA'?"
Processando questão 8/95: "Quais militares ativos são mentores de outros militares?"
Processando questão 9/95: "Quais tecnologias são dominadas por militares com patente de 1º Tenente?"
Processando questão 10/95: "Quais projetos requerem o conhecimento de Criptografia?"
Processand

Text-to-cypher-Gemma3-27B Cost: $1 Time:63m 51.1s

In [ ]:
run_evaluation()

Iniciando o pipeline de avaliação revisado...
Processando questão 1/95: "Quais militares com patente de Major possuem a competência 'Gestão de Projetos'?"
Processando questão 2/95: "Quais projetos utilizam a tecnologia PostgreSQL e estão em andamento?"
Processando questão 3/95: "Quais militares com patente de Coronel são responsáveis por documentação?"
Processando questão 4/95: "Quais projetos concluídos possuem documentação crítica de nível ALTA?"
Processando questão 5/95: "Quais militares possuem certificação PMP?"
Processando questão 6/95: "Quais treinamentos foram concluídos por militares de patente Capitão?"
Processando questão 7/95: "Quais competências são requeridas pelo projeto 'Simulador de Voo IA'?"
Processando questão 8/95: "Quais militares ativos são mentores de outros militares?"
Processando questão 9/95: "Quais tecnologias são dominadas por militares com patente de 1º Tenente?"
Processando questão 10/95: "Quais projetos requerem o conhecimento de Criptografia?"
Processand

27B + GPT-4o-mini

In [ ]:
run_evaluation()

Iniciando o pipeline de avaliação revisado...
Processando questão 1/95: "Quais militares com patente de Major possuem a competência 'Gestão de Projetos'?"
Processando questão 2/95: "Quais projetos utilizam a tecnologia PostgreSQL e estão em andamento?"
Processando questão 3/95: "Quais militares com patente de Coronel são responsáveis por documentação?"
Processando questão 4/95: "Quais projetos concluídos possuem documentação crítica de nível ALTA?"
Processando questão 5/95: "Quais militares possuem certificação PMP?"
Processando questão 6/95: "Quais treinamentos foram concluídos por militares de patente Capitão?"
Processando questão 7/95: "Quais competências são requeridas pelo projeto 'Simulador de Voo IA'?"
Processando questão 8/95: "Quais militares ativos são mentores de outros militares?"
Processando questão 9/95: "Quais tecnologias são dominadas por militares com patente de 1º Tenente?"
Processando questão 10/95: "Quais projetos requerem o conhecimento de Criptografia?"
Processand

ChatGPT 4 v2 Cost: $7.28 Time: 25m 45.0s

In [ ]:
run_evaluation()

Iniciando o pipeline de avaliação revisado...
Processando questão 1/95: "Quais Pessoas com patente de Major possuem a competência 'Gestão de Projetos'?"
Processando questão 2/95: "Quais projetos utilizam a tecnologia PostgreSQL e estão em andamento?"
Processando questão 3/95: "Quais Pessoas com patente de Coronel são responsáveis por documentação?"
Processando questão 4/95: "Quais projetos concluídos possuem documentação crítica de nível ALTA?"
Processando questão 5/95: "Quais Pessoas possuem certificação PMP?"
Processando questão 6/95: "Quais treinamentos foram concluídos por Pessoas de patente Capitão?"
Processando questão 7/95: "Quais competências são requeridas pelo projeto 'Simulador de Voo IA'?"
Processando questão 8/95: "Quais Pessoas ativos são mentores de outras Pessoas?"
Processando questão 9/95: "Quais tecnologias são dominadas por Pessoas com patente de 1º Tenente?"
Processando questão 10/95: "Quais projetos requerem o conhecimento de Criptografia?"
Processando questão 11/9

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: VINCULO_PROJETO_PESQUISA)} {position: line: 1, column: 50, offset: 49} for query: "MATCH (p:Pessoa)-[:TRABALHA_EM]->(pe:Pesquisa)-[:VINCULO_PROJETO_PESQUISA]->(pr:Projeto)\nWHERE toLower(pr.nomeProjeto) CONTAINS 'satcom defesa'\nRETURN p.nomePessoa"
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not avai

Processando questão 82/95: "Quais projetos utilizam Kafka?"
Processando questão 83/95: "Quais Pessoas possuem tanto certificação Cisco CCNA quanto competência de Programação?"
Processando questão 84/95: "Quais projetos concluídos possuem pesquisas vinculadas a C2?"
Processando questão 85/95: "Quais Pessoas de patente 3º Sargento são instrutores?"
Processando questão 86/95: "Quais tecnologias são consideradas Core nos projetos?"
Processando questão 87/95: "Quais Pessoas foram mentores desde 2024?"
Processando questão 88/95: "Quais projetos estão em andamento e utilizam MATLAB?"
Processando questão 89/95: "Quais Pessoas possuem graduação em Engenharia Aeronáutica?"
Processando questão 90/95: "Quais documentos foram publicados em 2022?"
Processando questão 91/95: "Quais Pessoas possuem nível avançado em Análise de Sinais?"
Processando questão 92/95: "Quais projetos estão planejados para 2025?"
Processando questão 93/95: "Quais Pessoas são responsáveis por documentação no Repositório A1?"


Text-to-cypher-Gemma3-4B v2 Cost: $ Time:

In [6]:
run_evaluation()

Iniciando o pipeline de avaliação revisado...
Processando questão 1/95: "Quais Pessoas com patente de Major possuem a competência 'Gestão de Projetos'?"
Processando questão 2/95: "Quais projetos utilizam a tecnologia PostgreSQL e estão em andamento?"
Processando questão 3/95: "Quais Pessoas com patente de Coronel são responsáveis por documentação?"
Processando questão 4/95: "Quais projetos concluídos possuem documentação crítica de nível ALTA?"
Processando questão 5/95: "Quais Pessoas possuem certificação PMP?"
Processando questão 6/95: "Quais treinamentos foram concluídos por Pessoas de patente Capitão?"
Processando questão 7/95: "Quais competências são requeridas pelo projeto 'Simulador de Voo IA'?"
Processando questão 8/95: "Quais Pessoas ativos são mentores de outras Pessoas?"
Processando questão 9/95: "Quais tecnologias são dominadas por Pessoas com patente de 1º Tenente?"
Processando questão 10/95: "Quais projetos requerem o conhecimento de Criptografia?"
Processando questão 11/9

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: descricaoConhecimento)} {position: line: 1, column: 539, offset: 538} for query: "MATCH (p:Projeto {nomeProjeto: 'Autopiloto Seguro'})-[:REQUER_CONHECIMENTO]->(c:Conhecimento)<-[:POSSUI_DOCUMENTACAO]-(d:Documentacao)-[:POSSUI_SUMARIO]->(s:Sumario)<-[:POSSUI_SUMARIO]-(pub:Publicacao)<-[:POSSUI_SUMARIO]-(sum:Sumario)<-[:POSSUI_SUMARIO]-(doc:Documentacao)<-[:POSSUI_DOCUMENTACAO]-(proj:Projeto)<-[:REQUER_COMPETENCIA]-(comp:Competencia)<-[:POSSUI_DOCUMENTACAO]-(proj2:Projeto)-[:REQUER_CONHECIMENTO]->(c2:Conhecimento)-[:POSSUI_DOCUMENTACAO]->(doc2:Documentaca

Processando questão 30/95: "Quais Pessoas com patente de Tenente-Coronel são mentores?"
Processando questão 31/95: "Quais documentos estão associados ao projeto Radar XBR-3?"
Processando questão 32/95: "Quais tecnologias foram certificadas pela certificação Cisco CCNA?"
Processando questão 33/95: "Quais Pessoas concluíram o treinamento em Aviónica?"
Processando questão 34/95: "Quais competências são críticas para o projeto C2 Unificado?"
Processando questão 35/95: "Quais Pessoas possuem publicações com palavras-chave 'ELINT'?"
Processando questão 36/95: "Quais processos possuem documentação armazenada no Hangar H1?"
Processando questão 37/95: "Quais projetos concluídos estão vinculados a pesquisas?"
Processando questão 38/95: "Quais Pessoas são cientistas de dados e dominam Neo4j?"
Processando questão 39/95: "Quais competências possuem avaliação avançada em Pessoas de patente Coronel?"
Processando questão 40/95: "Quais projetos possuem orçamento acima de 3 milhões?"
Processando questão

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: INSTITUICAO_TREINAMENTO)} {position: line: 1, column: 25, offset: 24} for query: "MATCH (t:Treinamento)-[:INSTITUICAO_TREINAMENTO]->(i:InstituicaoPosGraduacao) WHERE i.nomeInstituicao CONTAINS 'ITA' RETURN t"
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownLabelWarning} {category: UNRECOGNIZED} {title: The provided label is not in the database.} {description: One of the labels in your query is not available in the database, make sure you didn't misspell it or that the label is

Processando questão 46/95: "Quais documentos estão vinculados ao conhecimento de Manutenção Aeronáutica?"
Processando questão 47/95: "Quais Pessoas foram indicados para o treinamento Criptografia Aplicada?"
Processando questão 48/95: "Quais Pessoas de patente 2º Sargento possuem certificação CEH?"
Processando questão 49/95: "Quais projetos concluídos estão relacionados ao conhecimento de Redes Táticas?"
Processando questão 50/95: "Quais Pessoas com patente de Capitão possuem competência de Segurança da Informação?"
Processando questão 51/95: "Quais pesquisas estão ativas em 2025?"
Processando questão 52/95: "Quais Pessoas possuem pós-graduação em Gestão?"
Processando questão 53/95: "Quais projetos utilizam tecnologias de banco de dados?"
Processando questão 54/95: "Quais Pessoas publicaram em 2025?"
Processando questão 55/95: "Quais documentos estão vinculados ao processo Despacho de Voo?"
Processando questão 56/95: "Quais Pessoas ativos possuem certificações avançadas?"
Processando qu

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: POSSUI_COMPETENCIA)} {position: line: 1, column: 20, offset: 19} for query: 'MATCH (p:Pessoa)-[:POSSUI_COMPETENCIA]->(c:Competencia {nomeCompetencia: "Liderança"}) WHERE p.patentePessoa = "1º Sargento" RETURN p'
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you d

Processando questão 67/95: "Quais pesquisas foram concluídas em 2023?"
Processando questão 68/95: "Quais Pessoas com patente de Coronel possuem certificação PMP?"
Processando questão 69/95: "Quais projetos possuem documentação armazenada no Repositório A1?"
Processando questão 70/95: "Quais Pessoas dominam a tecnologia CUDA com nível igual ou superior a 4?"
Processando questão 71/95: "Quais projetos em andamento possuem orçamento inferior a 2 milhões?"
Processando questão 72/95: "Quais Pessoas de patente Major publicaram artigos?"
Processando questão 73/95: "Quais projetos requerem competência de Programação?"
Processando questão 74/95: "Quais treinamentos foram realizados em modalidade EAD?"
Processando questão 75/95: "Quais Pessoas concluíram a graduação no ITA?"
Processando questão 76/95: "Quais projetos utilizam a tecnologia OpenCV?"
Processando questão 77/95: "Quais Pessoas ativos possuem conhecimento em IA/ML?"
Processando questão 78/95: "Quais documentos foram atualizados após 2

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: VINCLUADO_A)} {position: line: 1, column: 49, offset: 48} for query: 'MATCH (p:Pessoa)-[:TRABALHA_EM]->(r:Pesquisa)-[:VINCLUADO_A]->(pr:Projeto {nomeProjeto: "SatCom Defesa"}) RETURN p.nomePessoa'
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell 

Processando questão 82/95: "Quais projetos utilizam Kafka?"
Processando questão 83/95: "Quais Pessoas possuem tanto certificação Cisco CCNA quanto competência de Programação?"
Processando questão 84/95: "Quais projetos concluídos possuem pesquisas vinculadas a C2?"
Processando questão 85/95: "Quais Pessoas de patente 3º Sargento são instrutores?"
Processando questão 86/95: "Quais tecnologias são consideradas Core nos projetos?"
Processando questão 87/95: "Quais Pessoas foram mentores desde 2024?"
Processando questão 88/95: "Quais projetos estão em andamento e utilizam MATLAB?"
Processando questão 89/95: "Quais Pessoas possuem graduação em Engenharia Aeronáutica?"
Processando questão 90/95: "Quais documentos foram publicados em 2022?"
Processando questão 91/95: "Quais Pessoas possuem nível avançado em Análise de Sinais?"
Processando questão 92/95: "Quais projetos estão planejados para 2025?"
Processando questão 93/95: "Quais Pessoas são responsáveis por documentação no Repositório A1?"


Text-to-cypher-4B + ChatGPT 4 Cost: 2.63 $ Time: 10m 37.1s

In [7]:
run_evaluation()

Iniciando o pipeline de avaliação revisado...
Processando questão 1/95: "Quais Pessoas com patente de Major possuem a competência 'Gestão de Projetos'?"
Processando questão 2/95: "Quais projetos utilizam a tecnologia PostgreSQL e estão em andamento?"
Processando questão 3/95: "Quais Pessoas com patente de Coronel são responsáveis por documentação?"
Processando questão 4/95: "Quais projetos concluídos possuem documentação crítica de nível ALTA?"
Processando questão 5/95: "Quais Pessoas possuem certificação PMP?"
Processando questão 6/95: "Quais treinamentos foram concluídos por Pessoas de patente Capitão?"
Processando questão 7/95: "Quais competências são requeridas pelo projeto 'Simulador de Voo IA'?"
Processando questão 8/95: "Quais Pessoas ativos são mentores de outras Pessoas?"
Processando questão 9/95: "Quais tecnologias são dominadas por Pessoas com patente de 1º Tenente?"
Processando questão 10/95: "Quais projetos requerem o conhecimento de Criptografia?"
Processando questão 11/9

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: descricaoConhecimento)} {position: line: 1, column: 539, offset: 538} for query: "MATCH (p:Projeto {nomeProjeto: 'Autopiloto Seguro'})-[:REQUER_CONHECIMENTO]->(c:Conhecimento)<-[:POSSUI_DOCUMENTACAO]-(d:Documentacao)-[:POSSUI_SUMARIO]->(s:Sumario)<-[:POSSUI_SUMARIO]-(pub:Publicacao)<-[:POSSUI_SUMARIO]-(sum:Sumario)<-[:POSSUI_SUMARIO]-(doc:Documentacao)<-[:POSSUI_DOCUMENTACAO]-(proj:Projeto)<-[:REQUER_COMPETENCIA]-(comp:Competencia)<-[:POSSUI_DOCUMENTACAO]-(proj2:Projeto)-[:REQUER_CONHECIMENTO]->(c2:Conhecimento)-[:POSSUI_DOCUMENTACAO]->(doc2:Documentaca

Processando questão 30/95: "Quais Pessoas com patente de Tenente-Coronel são mentores?"
Processando questão 31/95: "Quais documentos estão associados ao projeto Radar XBR-3?"
Processando questão 32/95: "Quais tecnologias foram certificadas pela certificação Cisco CCNA?"
Processando questão 33/95: "Quais Pessoas concluíram o treinamento em Aviónica?"
Processando questão 34/95: "Quais competências são críticas para o projeto C2 Unificado?"
Processando questão 35/95: "Quais Pessoas possuem publicações com palavras-chave 'ELINT'?"
Processando questão 36/95: "Quais processos possuem documentação armazenada no Hangar H1?"
Processando questão 37/95: "Quais projetos concluídos estão vinculados a pesquisas?"
Processando questão 38/95: "Quais Pessoas são cientistas de dados e dominam Neo4j?"
Processando questão 39/95: "Quais competências possuem avaliação avançada em Pessoas de patente Coronel?"
Processando questão 40/95: "Quais projetos possuem orçamento acima de 3 milhões?"
Processando questão

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: INSTITUICAO_TREINAMENTO)} {position: line: 1, column: 25, offset: 24} for query: "MATCH (t:Treinamento)-[:INSTITUICAO_TREINAMENTO]->(i:InstituicaoPosGraduacao) WHERE i.nomeInstituicao CONTAINS 'ITA' RETURN t"
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownLabelWarning} {category: UNRECOGNIZED} {title: The provided label is not in the database.} {description: One of the labels in your query is not available in the database, make sure you didn't misspell it or that the label is

Processando questão 46/95: "Quais documentos estão vinculados ao conhecimento de Manutenção Aeronáutica?"
Processando questão 47/95: "Quais Pessoas foram indicados para o treinamento Criptografia Aplicada?"
Processando questão 48/95: "Quais Pessoas de patente 2º Sargento possuem certificação CEH?"
Processando questão 49/95: "Quais projetos concluídos estão relacionados ao conhecimento de Redes Táticas?"
Processando questão 50/95: "Quais Pessoas com patente de Capitão possuem competência de Segurança da Informação?"
Processando questão 51/95: "Quais pesquisas estão ativas em 2025?"
Processando questão 52/95: "Quais Pessoas possuem pós-graduação em Gestão?"
Processando questão 53/95: "Quais projetos utilizam tecnologias de banco de dados?"
Processando questão 54/95: "Quais Pessoas publicaram em 2025?"
Processando questão 55/95: "Quais documentos estão vinculados ao processo Despacho de Voo?"
Processando questão 56/95: "Quais Pessoas ativos possuem certificações avançadas?"
Processando qu

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: POSSUI_COMPETENCIA)} {position: line: 1, column: 20, offset: 19} for query: 'MATCH (p:Pessoa)-[:POSSUI_COMPETENCIA]->(c:Competencia {nomeCompetencia: "Liderança"}) WHERE p.patentePessoa = "1º Sargento" RETURN p'
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you d

Processando questão 67/95: "Quais pesquisas foram concluídas em 2023?"
Processando questão 68/95: "Quais Pessoas com patente de Coronel possuem certificação PMP?"
Processando questão 69/95: "Quais projetos possuem documentação armazenada no Repositório A1?"
Processando questão 70/95: "Quais Pessoas dominam a tecnologia CUDA com nível igual ou superior a 4?"
Processando questão 71/95: "Quais projetos em andamento possuem orçamento inferior a 2 milhões?"
Processando questão 72/95: "Quais Pessoas de patente Major publicaram artigos?"
Processando questão 73/95: "Quais projetos requerem competência de Programação?"
Processando questão 74/95: "Quais treinamentos foram realizados em modalidade EAD?"
Processando questão 75/95: "Quais Pessoas concluíram a graduação no ITA?"
Processando questão 76/95: "Quais projetos utilizam a tecnologia OpenCV?"
Processando questão 77/95: "Quais Pessoas ativos possuem conhecimento em IA/ML?"
Processando questão 78/95: "Quais documentos foram atualizados após 2

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: VINCLUADO_A)} {position: line: 1, column: 49, offset: 48} for query: 'MATCH (p:Pessoa)-[:TRABALHA_EM]->(r:Pesquisa)-[:VINCLUADO_A]->(pr:Projeto {nomeProjeto: "SatCom Defesa"}) RETURN p.nomePessoa'
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell 

Processando questão 82/95: "Quais projetos utilizam Kafka?"
Processando questão 83/95: "Quais Pessoas possuem tanto certificação Cisco CCNA quanto competência de Programação?"
Processando questão 84/95: "Quais projetos concluídos possuem pesquisas vinculadas a C2?"
Processando questão 85/95: "Quais Pessoas de patente 3º Sargento são instrutores?"
Processando questão 86/95: "Quais tecnologias são consideradas Core nos projetos?"
Processando questão 87/95: "Quais Pessoas foram mentores desde 2024?"
Processando questão 88/95: "Quais projetos estão em andamento e utilizam MATLAB?"
Processando questão 89/95: "Quais Pessoas possuem graduação em Engenharia Aeronáutica?"
Processando questão 90/95: "Quais documentos foram publicados em 2022?"
Processando questão 91/95: "Quais Pessoas possuem nível avançado em Análise de Sinais?"
Processando questão 92/95: "Quais projetos estão planejados para 2025?"
Processando questão 93/95: "Quais Pessoas são responsáveis por documentação no Repositório A1?"


Text-to-cypher-27B v2

In [13]:
run_evaluation()

Iniciando o pipeline de avaliação revisado...
Processando questão 1/95: "Quais Pessoas com patente de Major possuem a competência 'Gestão de Projetos'?"
Processando questão 2/95: "Quais projetos utilizam a tecnologia PostgreSQL e estão em andamento?"
Processando questão 3/95: "Quais Pessoas com patente de Coronel são responsáveis por documentação?"
Processando questão 4/95: "Quais projetos concluídos possuem documentação crítica de nível ALTA?"
Processando questão 5/95: "Quais Pessoas possuem certificação PMP?"
Processando questão 6/95: "Quais treinamentos foram concluídos por Pessoas de patente Capitão?"
Processando questão 7/95: "Quais competências são requeridas pelo projeto 'Simulador de Voo IA'?"
Processando questão 8/95: "Quais Pessoas ativos são mentores de outras Pessoas?"
Processando questão 9/95: "Quais tecnologias são dominadas por Pessoas com patente de 1º Tenente?"
Processando questão 10/95: "Quais projetos requerem o conhecimento de Criptografia?"
Processando questão 11/9

KeyError: 'factual_consistency'

In [ ]:
df = pd.read_csv(RESULTS_CSV_PATH)
cypher_exec_rate = df['cypher_executable'].mean() * 100
average_bert_score = df['bert_score_f1'].mean()

with open(QA_FILE_PATH, 'r', encoding='utf-8') as f:
    qa_dataset = json.load(f)

total_items = len(qa_dataset)



results_consistency = []

for i, item in enumerate(qa_dataset):
    print(f"Processando questão {i+1}/{total_items}: \"{item['question']}\"")
    question = item['question']
    gold_cypher = item['gold_cypher']
    gold_answer = item['gold_answer']
    factual_consistency = evaluate_factual_consistency(question, gold_answer, df['generated_answer'][i], JUDGE_LLM)
    results_consistency.append(factual_consistency)
results_consistency_df = pd.DataFrame(results_consistency, columns=['factual_consistency'])
consistency_distribution = results_consistency_df['factual_consistency'].value_counts(normalize=True) * 100
print("\nDistribuição da Consistência Factual:")
print(consistency_distribution.to_string())
print("\n--- RESUMO DA AVALIAÇÃO REVISADA ---")
print(f"Taxa de Execução das Queries Cypher: {cypher_exec_rate:.2f}%")
print(f"BERTScore F1 Médio (Similaridade Semântica): {average_bert_score:.4f}")

O Coronel Militar 70 (email: militar70@fab.mil.br) é responsável pela documentação 'Guia de Manutenção' (DOC-003), localizada no Hangar H1.
Processando questão 1/95: "Quais Pessoas com patente de Major possuem a competência 'Gestão de Projetos'?"
Processando questão 2/95: "Quais projetos utilizam a tecnologia PostgreSQL e estão em andamento?"
Processando questão 3/95: "Quais Pessoas com patente de Coronel são responsáveis por documentação?"
Processando questão 4/95: "Quais projetos concluídos possuem documentação crítica de nível ALTA?"
Processando questão 5/95: "Quais Pessoas possuem certificação PMP?"
Processando questão 6/95: "Quais treinamentos foram concluídos por Pessoas de patente Capitão?"
Processando questão 7/95: "Quais competências são requeridas pelo projeto 'Simulador de Voo IA'?"
Processando questão 8/95: "Quais Pessoas ativos são mentores de outras Pessoas?"
Processando questão 9/95: "Quais tecnologias são dominadas por Pessoas com patente de 1º Tenente?"
Processando que

Vamos usar os resultados cypher gerados pelo modelo 27B quantizado com a extração de resposta potente do GPT-4

In [35]:
import time
from openai import RateLimitError

df = pd.read_csv(RESULTS_CSV_PATH)
cypher_exec_rate = df['cypher_executable'].mean() * 100

with open(QA_FILE_PATH, 'r', encoding='utf-8') as f:
    qa_dataset = json.load(f)

total_items = len(qa_dataset)

# --- Create chain with GPT-4 and run

gpt,_,_ = CreateLLMOpenai(openai_api_key)
chainGPT = CreateChain(graph,gpt,cypher_prompt,qa_prompt)

# --- Connect to Neo4j and run a test query

neo4j_driver = GraphDatabase.driver(uri, auth=(username, password))

evaluation_results = []

for i, item in enumerate(qa_dataset):
    print(f"Processando questão {i+1}/{total_items}: \"{item['question']}\"")
    question = item['question']
    gold_cypher = item['gold_cypher']
    gold_answer = item['gold_answer']

    context = neo4j_driver.session().run(df['generated_cypher'][i]).data()
    generated_answer = None
    for attempt in range(5):  # tenta no máximo 5 vezes
        try:
            generated_answer = chainGPT.qa_chain.invoke(
                {"context": context, "query": question}
            )
            break
        except RateLimitError as e:
            wait_time = (2 ** attempt) * 0.5  # backoff exponencial
            print(f"⚠️ RateLimitError na questão {i+1}, tentando novamente em {wait_time:.2f}s...")
            time.sleep(wait_time)
    if generated_answer is None:
        print(f"❌ Não foi possível obter resposta para a questão {i+1}, pulando...")
        continue

    # 2. Avaliar a resposta textual
    P, R, F1 = bert_scorer([generated_answer], [gold_answer], lang="pt", model_type='bert-base-multilingual-cased')
    bert_f1 = F1.item()
    factual_consistency = evaluate_factual_consistency(question, gold_answer, generated_answer, JUDGE_LLM)
    evaluation_results.append({
        "id": item['id'],
        "question": question,
        "gold_cypher": gold_cypher,
        "generated_cypher": df['generated_cypher'][i],
        "cypher_executable": df['cypher_executable'][i],
        "gold_answer": gold_answer,
        "generated_answer": generated_answer,
        "bert_score_f1": bert_f1,
        "factual_consistency": factual_consistency
    })

neo4j_driver.close()
df_results = pd.DataFrame(evaluation_results)
df_results.to_csv("evaluation_results_revised-27B+GPT.csv", index=False, encoding='utf-8-sig')
print(f"\nResultados detalhados salvos em: evaluation_results_revised-27B+GPT.csv")
# --- ANÁLISE DOS RESULTADOS ---
cypher_exec_rate = df_results['cypher_executable'].mean() * 100
average_bert_score = df_results['bert_score_f1'].mean()
consistency_distribution = df_results['factual_consistency'].value_counts(normalize=True) * 100

print("\n--- RESUMO DA AVALIAÇÃO REVISADA ---")
print(f"Taxa de Execução das Queries Cypher: {cypher_exec_rate:.2f}%")
print(f"BERTScore F1 Médio (Similaridade Semântica): {average_bert_score:.4f}")
print("\nDistribuição da Consistência Factual:")
print(consistency_distribution.to_string())
print("-------------------------------------\n")

print("Amostra de Respostas marcadas como INCONSISTENTES:")
inconsistent_df = df[df['factual_consistency'] == 'INCONSISTENTE']
print(inconsistent_df[['id', 'question', 'gold_answer', 'generated_answer']].head())

Processando questão 1/95: "Quais Pessoas com patente de Major possuem a competência 'Gestão de Projetos'?"
Processando questão 2/95: "Quais projetos utilizam a tecnologia PostgreSQL e estão em andamento?"
Processando questão 3/95: "Quais Pessoas com patente de Coronel são responsáveis por documentação?"
Processando questão 4/95: "Quais projetos concluídos possuem documentação crítica de nível ALTA?"
Processando questão 5/95: "Quais Pessoas possuem certificação PMP?"
Processando questão 6/95: "Quais treinamentos foram concluídos por Pessoas de patente Capitão?"
Processando questão 7/95: "Quais competências são requeridas pelo projeto 'Simulador de Voo IA'?"
Processando questão 8/95: "Quais Pessoas ativos são mentores de outras Pessoas?"
Processando questão 9/95: "Quais tecnologias são dominadas por Pessoas com patente de 1º Tenente?"
Processando questão 10/95: "Quais projetos requerem o conhecimento de Criptografia?"
Processando questão 11/95: "Quais Pessoas possuem publicações em confe

KeyError: 'factual_consistency'

Teste

In [11]:
from neo4j.time import Date

def serialize_neo4j(obj):
    if isinstance(obj, Date):
        return obj.iso_format()  # transforma Date -> "YYYY-MM-DD"
    if isinstance(obj, dict):
        return {k: serialize_neo4j(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [serialize_neo4j(v) for v in obj]
    return obj

import time
from openai import RateLimitError

df = pd.read_csv(RESULTS_CSV_PATH)
cypher_exec_rate = df['cypher_executable'].mean() * 100

with open(QA_FILE_PATH, 'r', encoding='utf-8') as f:
    qa_dataset = json.load(f)

total_items = len(qa_dataset)

# --- Create chain with GPT-4 and run

gpt,_,_ = CreateLLMOpenai(openai_api_key)
chainGPT = CreateChain(graph,gpt,cypher_prompt,qa_prompt)

# --- Connect to Neo4j and run a test query

neo4j_driver = GraphDatabase.driver(uri, auth=(username, password))

evaluation_results = []

for i, item in enumerate(qa_dataset):
    print(f"Processando questão {i+1}/{total_items}: \"{item['question']}\"")
    question = item['question']
    gold_cypher = item['gold_cypher']
    gold_answer = item['gold_answer']

    context = neo4j_driver.session().run(df['generated_cypher'][i]).data()
    context_serializable = serialize_neo4j(context)
    context_json = json.dumps(context_serializable, ensure_ascii=False, indent=2)
    generated_answer = None
    for attempt in range(5):  # tenta no máximo 5 vezes
        try:
            generated_answer = chainGPT.qa_chain.invoke(
                {"context": context_json, "query": question}
            )
            break
        except RateLimitError as e:
            wait_time = (2 ** attempt) * 0.5  # backoff exponencial
            print(f"⚠️ RateLimitError na questão {i+1}, tentando novamente em {wait_time:.2f}s...")
            time.sleep(wait_time)
    if generated_answer is None:
        print(f"❌ Não foi possível obter resposta para a questão {i+1}, pulando...")
        continue

    # 2. Avaliar a resposta textual
    P, R, F1 = bert_scorer([generated_answer], [gold_answer], lang="pt", model_type='bert-base-multilingual-cased')
    bert_f1 = F1.item()
    factual_consistency = evaluate_factual_consistency(question, gold_answer, generated_answer, JUDGE_LLM)
    evaluation_results.append({
        "id": item['id'],
        "question": question,
        "gold_cypher": gold_cypher,
        "generated_cypher": df['generated_cypher'][i],
        "cypher_executable": df['cypher_executable'][i],
        "gold_answer": gold_answer,
        "generated_answer": generated_answer,
        "bert_score_f1": bert_f1,
        "factual_consistency": factual_consistency
    })

neo4j_driver.close()
df_results = pd.DataFrame(evaluation_results)
df_results.to_csv("evaluation_results_revised-27B+GPT4.1-mini.csv", index=False, encoding='utf-8-sig')
print(f"\nResultados detalhados salvos em: evaluation_results_revised-27B+GPT4.1-mini.csv")
# --- ANÁLISE DOS RESULTADOS ---
cypher_exec_rate = df_results['cypher_executable'].mean() * 100
average_bert_score = df_results['bert_score_f1'].mean()
consistency_distribution = df_results['factual_consistency'].value_counts(normalize=True) * 100

print("\n--- RESUMO DA AVALIAÇÃO REVISADA ---")
print(f"Taxa de Execução das Queries Cypher: {cypher_exec_rate:.2f}%")
print(f"BERTScore F1 Médio (Similaridade Semântica): {average_bert_score:.4f}")
print("\nDistribuição da Consistência Factual:")
print(consistency_distribution.to_string())
print("-------------------------------------\n")

Processando questão 1/95: "Quais Pessoas com patente de Major possuem a competência 'Gestão de Projetos'?"
Processando questão 2/95: "Quais projetos utilizam a tecnologia PostgreSQL e estão em andamento?"
Processando questão 3/95: "Quais Pessoas com patente de Coronel são responsáveis por documentação?"
Processando questão 4/95: "Quais projetos concluídos possuem documentação crítica de nível ALTA?"
Processando questão 5/95: "Quais Pessoas possuem certificação PMP?"
Processando questão 6/95: "Quais treinamentos foram concluídos por Pessoas de patente Capitão?"
Processando questão 7/95: "Quais competências são requeridas pelo projeto 'Simulador de Voo IA'?"
Processando questão 8/95: "Quais Pessoas ativos são mentores de outras Pessoas?"
Processando questão 9/95: "Quais tecnologias são dominadas por Pessoas com patente de 1º Tenente?"
Processando questão 10/95: "Quais projetos requerem o conhecimento de Criptografia?"
Processando questão 11/95: "Quais Pessoas possuem publicações em confe